# Notebook 6 — Pourquoi STT_ISCH s'améliore-t-il avec la dérivation I seule ? (CPU)

**Réglages :** *Accelerator* **None** ; *Internet* **On** (pour télécharger `ptbxl_database.csv`,
13 Mo, depuis PhysioNet — sans Internet, la partie C est simplement sautée).
*Inputs* : le dataset du code, la sortie du notebook 4 (runs) **et** la sortie du notebook 1 (cache, qui
contient `ptbxl/meta.csv`).

Le notebook n'entraîne rien : il relit les prédictions `no_text` et `leadI_no_text` déjà sauvegardées.
Résultats dans `/kaggle/working/stt_check/` : trois tableaux CSV et la figure `figS5_stt_leadI_check`.

In [ ]:
import glob, os, re, shutil
def _version(init):
    m = re.search(r'__version__\s*=\s*"([^"]+)"', open(init).read())
    return m.group(1) if m else "0"
# plusieurs copies du code peuvent être présentes (dataset du code, sorties d'anciens notebooks) :
# on prend la plus récente
hits = sorted(glob.glob("/kaggle/input/**/ecgfusion/__init__.py", recursive=True), key=_version)
assert hits, "Code introuvable : ajoutez le dataset contenant ecg_fusion.zip via « Add Input »."
src = os.path.dirname(os.path.dirname(hits[-1]))
shutil.rmtree("/tmp/ecg_fusion", ignore_errors=True)
shutil.copytree(src, "/tmp/ecg_fusion")   # hors /kaggle/working : le code n'encombre pas la sortie
os.chdir("/tmp/ecg_fusion")
!pip install -q wfdb
print(f"Code version {_version(hits[-1])} prêt (source : {src})")

In [ ]:
RUN_NAMES = ["v2_amplitude", "v2_amp_seed1", "v2_amp_seed2"]   # les 3 graines du modèle final
runs = []
for n in RUN_NAMES:
    hits = sorted(glob.glob(f"/kaggle/input/**/runs/{n}/eval/summary.csv", recursive=True))
    if hits:
        runs.append(os.path.dirname(os.path.dirname(hits[0])))
    else:
        print("introuvable :", n)
assert runs, "Aucun run trouvé : ajoutez la sortie du notebook 4 en Input."
metas = sorted(glob.glob("/kaggle/input/**/ptbxl/meta.csv", recursive=True))
CACHE = os.path.dirname(metas[0]) if metas else ""
print("Runs :", runs); print("Cache PTB-XL :", CACHE or "introuvable (partie C sautée)")
cmd = f"python scripts/stt_lead_check.py --runs {' '.join(runs)} --out /kaggle/working/stt_check"
if CACHE:
    cmd += f" --ptbxl_cache {CACHE}"
!{cmd}

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/stt_check/figS5_stt_leadI_check.png", width=800))